In [1]:
import pandas as pd
import numpy as np
import sys, os
from importlib import reload
import json
from pathlib import Path
from dataclasses import asdict

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from analysis import plots
import analysis.report
import training.gbm_model_trainer
from training.gbm_model_trainer import GBMModelTrainerConfig 

reload(analysis)
reload(analysis.plots)
reload(analysis.report)
reload(training.gbm_model_trainer)

<module 'training.gbm_model_trainer' from '/home/sagemaker-user/analysis-tools/src/training/gbm_model_trainer.py'>

## Load data

In [2]:
df = pd.read_csv("../data/bank-full.csv", delimiter=';')

In [3]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [4]:
# Convert 'y' to numeric for analysis
df["target"] = np.where(df["y"] == "no", 0, 1)

# Add weights to test weight functionality of plot_target_vs_predictors()
df["weights"] = np.abs(np.random.randn(len(df.index)))

# Add an arbitrary data split for testing functions
df["random"] = np.random.uniform(0, 1, len(df.index))
df["split"] = np.where(
    df["random"]  > 0.80, 
    "H", 
    np.where(
        df["random"]  > 0.50, 
        "V", 
        "T"
    )
)

In [5]:
df["split"].value_counts(dropna=False, normalize=True)

split
T    0.496008
V    0.302714
H    0.201278
Name: proportion, dtype: float64

In [6]:
df.groupby("split")["target"].mean()

split
H    0.117143
T    0.115942
V    0.118588
Name: target, dtype: float64

In [7]:
# Split data; define features and target
train = df.query("split == 'T'").drop(columns=["y", "split"])
test = df.query("split == 'V'").drop(columns=["y", "split"])
holdout = df.query("split == 'H'").drop(columns=["y", "split"])

## Use ModelTrainer class for hyperparameter tuning
- Test hyperparameter tuning

### Search over spaces

In [8]:
reload(training.gbm_model_trainer)
reload(analysis.report)
reload(analysis.plots)

<module 'analysis.plots' from '/home/sagemaker-user/analysis-tools/src/analysis/plots.py'>

In [9]:
from training.gbm_model_trainer import GBMModelTrainer
from xgboost import XGBClassifier

mt_xgboost = GBMModelTrainer(
    model_class=XGBClassifier,
    config_path="../examples/training_config_example_xgboost.yaml",
    train_df=train,
    valid_df=test,
    holdout_df=holdout
)

In [10]:
mt_xgboost.config

GBMModelTrainerConfig(actual_col='target', predicted_col='pred', output_dir='outputs', log_file='training.log', report_file='model_analysis.html', hyperparameters={'colsample_bytree': 0.5, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'objective': 'binary:logistic', 'random_state': 42, 'subsample': 0.5}, output_log=True, output_report=True, report_params={}, plots_to_add=[{'plot': 'plot_error_by_group_grid', 'title': 'Error by Analysis Variables', 'kwargs': {'group_cols': ['job', 'education', 'age']}}, {'plot': 'gain_curve_with_gini', 'title': 'Gain Curve / Lorenz Curve'}, {'plot': 'partial_gini_plot', 'title': 'Partial Gini (Top 15%)', 'kwargs': {'top_percent': 15}}, {'plot': 'lift_chart', 'title': 'Lift Chart'}, {'plot': 'crunched_residual_plot', 'title': 'Crunched Residuals'}, {'plot': 'plot_residual_fit', 'title': 'Std and Avg of Residuals', 'kwargs': {'residual_type': 'normalized'}}], tabulate_vars=['job', 'marital', 'education'], tuning=TuningConfig(search_space={'ma

In [13]:
tuned_df = mt_xgboost.tune()
tuned_df.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_booster,param_learning_rate,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score,rank_train_score
18,0.205039,0.000642,0.010564,0.000114,gbtree,0.021554,3,"{'booster': 'gbtree', 'learning_rate': 0.02155...",0.774316,0.565473,0.820809,0.720200,0.111042,1,0.845633,0.912030,0.858955,0.872206,0.028680,17
11,0.208317,0.002254,0.010540,0.000131,gbtree,0.019915,3,"{'booster': 'gbtree', 'learning_rate': 0.01991...",0.772192,0.567226,0.819668,0.719695,0.109540,2,0.843217,0.910261,0.857943,0.870474,0.028769,18
14,20.922568,0.050583,0.136355,0.000023,dart,0.018715,3,"{'booster': 'dart', 'learning_rate': 0.0187151...",0.767032,0.564382,0.820547,0.717320,0.110329,3,0.838811,0.908591,0.857125,0.868176,0.029539,19
7,19.989689,0.030499,0.138000,0.001119,dart,0.03044,3,"{'booster': 'dart', 'learning_rate': 0.0304399...",0.775291,0.546713,0.823658,0.715221,0.120778,4,0.855208,0.918161,0.866548,0.879972,0.027398,16
10,0.212065,0.013857,0.010807,0.000377,gbtree,0.01,3,"{'booster': 'gbtree', 'learning_rate': 0.01, '...",0.760743,0.554157,0.810587,0.708496,0.111015,5,0.824470,0.897828,0.844909,0.855736,0.030911,20


In [14]:
# Train using tuned hyperparameters
mt_xgboost.train()

✅ Analysis report generated at outputs/model_analysis.html


## Test with CAT Boost

In [15]:
from catboost import CatBoostClassifier
mt_catboost = GBMModelTrainer(
    model_class=CatBoostClassifier,
    config_path="../examples/training_config_example_catboost.yaml",
    train_df=train,
    valid_df=test,
    holdout_df=holdout
)

In [16]:
mt_catboost.config

GBMModelTrainerConfig(actual_col='target', predicted_col='pred', output_dir='outputs', log_file='training.log', report_file='model_analysis.html', hyperparameters={'n_estimators': 200}, output_log=True, output_report=True, report_params={}, plots_to_add=[{'plot': 'plot_error_by_group_grid', 'title': 'Error by Analysis Variables', 'kwargs': {'group_cols': ['job', 'education', 'age']}}, {'plot': 'gain_curve_with_gini', 'title': 'Gain Curve / Lorenz Curve'}, {'plot': 'partial_gini_plot', 'title': 'Partial Gini (Top 15%)', 'kwargs': {'top_percent': 15}}, {'plot': 'lift_chart', 'title': 'Lift Chart'}, {'plot': 'crunched_residual_plot', 'title': 'Crunched Residuals'}, {'plot': 'plot_residual_fit', 'title': 'Std and Avg of Residuals', 'kwargs': {'residual_type': 'normalized'}}], tabulate_vars=['job', 'marital', 'education'], tuning=TuningConfig(search_space={'depth': Integer(low=3, high=10, prior='uniform', transform='identity'), 'bagging_temperature': Real(low=0.01, high=0.95, prior='uniform

In [19]:
mt_catboost.tune()

0:	learn: 0.6673295	total: 11.9ms	remaining: 2.37s
1:	learn: 0.6425595	total: 20.9ms	remaining: 2.07s
2:	learn: 0.6195463	total: 30.4ms	remaining: 2s
3:	learn: 0.6014448	total: 37.9ms	remaining: 1.86s
4:	learn: 0.5838326	total: 47.6ms	remaining: 1.85s
5:	learn: 0.5608441	total: 56ms	remaining: 1.81s
6:	learn: 0.5438229	total: 64.2ms	remaining: 1.77s
7:	learn: 0.5263321	total: 71.9ms	remaining: 1.73s
8:	learn: 0.5128891	total: 81.6ms	remaining: 1.73s
9:	learn: 0.4992749	total: 90.6ms	remaining: 1.72s
10:	learn: 0.4867353	total: 98.2ms	remaining: 1.69s
11:	learn: 0.4747716	total: 107ms	remaining: 1.67s
12:	learn: 0.4623341	total: 117ms	remaining: 1.68s
13:	learn: 0.4520853	total: 127ms	remaining: 1.69s
14:	learn: 0.4423056	total: 136ms	remaining: 1.68s
15:	learn: 0.4353082	total: 141ms	remaining: 1.62s
16:	learn: 0.4287936	total: 145ms	remaining: 1.56s
17:	learn: 0.4210965	total: 155ms	remaining: 1.57s
18:	learn: 0.4127370	total: 164ms	remaining: 1.56s
19:	learn: 0.4060943	total: 173ms	r

/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/skopt/optimizer/optimizer.py:449: UserWarning: The objective has been evaluated at this point before.
  warnings.warn("The objective has been evaluated "


0:	learn: 0.6748793	total: 5.76ms	remaining: 1.15s
1:	learn: 0.6585994	total: 11.2ms	remaining: 1.11s
2:	learn: 0.6423565	total: 15.4ms	remaining: 1.01s
3:	learn: 0.6278609	total: 18.8ms	remaining: 920ms
4:	learn: 0.6127578	total: 23.1ms	remaining: 899ms
5:	learn: 0.5998262	total: 28.3ms	remaining: 916ms
6:	learn: 0.5862093	total: 32.4ms	remaining: 893ms
7:	learn: 0.5746153	total: 36.5ms	remaining: 876ms
8:	learn: 0.5636636	total: 39.7ms	remaining: 842ms
9:	learn: 0.5532809	total: 42.2ms	remaining: 802ms
10:	learn: 0.5421070	total: 46.7ms	remaining: 803ms
11:	learn: 0.5324934	total: 50.4ms	remaining: 789ms
12:	learn: 0.5223314	total: 53.2ms	remaining: 765ms
13:	learn: 0.5127099	total: 56.3ms	remaining: 748ms
14:	learn: 0.5046508	total: 60.3ms	remaining: 744ms
15:	learn: 0.4970765	total: 63.8ms	remaining: 734ms
16:	learn: 0.4898709	total: 65.2ms	remaining: 702ms
17:	learn: 0.4830603	total: 69.1ms	remaining: 698ms
18:	learn: 0.4755619	total: 72.5ms	remaining: 691ms
19:	learn: 0.4693490	t

/home/sagemaker-user/.conda/envs/NY_TIER_V140_SAGEMAKER/lib/python3.11/site-packages/skopt/optimizer/optimizer.py:449: UserWarning: The objective has been evaluated at this point before.
  warnings.warn("The objective has been evaluated "


0:	learn: 0.6748793	total: 5.47ms	remaining: 1.09s
1:	learn: 0.6585994	total: 10.6ms	remaining: 1.05s
2:	learn: 0.6423565	total: 14.7ms	remaining: 967ms
3:	learn: 0.6278609	total: 18.4ms	remaining: 900ms
4:	learn: 0.6127578	total: 22.4ms	remaining: 875ms
5:	learn: 0.5998262	total: 26.9ms	remaining: 871ms
6:	learn: 0.5862093	total: 31.4ms	remaining: 864ms
7:	learn: 0.5746153	total: 35.5ms	remaining: 852ms
8:	learn: 0.5636636	total: 38.5ms	remaining: 817ms
9:	learn: 0.5532809	total: 41ms	remaining: 779ms
10:	learn: 0.5421070	total: 45.8ms	remaining: 787ms
11:	learn: 0.5324934	total: 49.4ms	remaining: 774ms
12:	learn: 0.5223314	total: 52.2ms	remaining: 750ms
13:	learn: 0.5127099	total: 55.4ms	remaining: 736ms
14:	learn: 0.5046508	total: 59.4ms	remaining: 733ms
15:	learn: 0.4970765	total: 62.5ms	remaining: 719ms
16:	learn: 0.4898709	total: 63.9ms	remaining: 687ms
17:	learn: 0.4830603	total: 67.8ms	remaining: 685ms
18:	learn: 0.4755619	total: 71.2ms	remaining: 678ms
19:	learn: 0.4693490	tot

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_bagging_temperature,param_depth,param_l2_leaf_reg,param_leaf_estimation_iterations,param_random_strength,params,...,split2_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,mean_train_score,std_train_score,rank_train_score
13,1.085619,0.015045,0.007182,0.000118,0.178914,3,3.381304,3,18.341148,"{'bagging_temperature': 0.17891372112276038, '...",...,0.799451,0.740824,0.104159,1,0.800328,0.874610,0.820977,0.831972,0.031306,23
4,1.125236,0.012505,0.007485,0.000281,0.528878,3,9.321459,4,19.240633,"{'bagging_temperature': 0.5288777012609818, 'd...",...,0.798788,0.736715,0.103583,2,0.802229,0.877894,0.817718,0.832614,0.032636,21
20,0.776718,0.015088,0.006970,0.000695,0.01,3,10.0,1,20.0,"{'bagging_temperature': 0.01, 'depth': 3, 'l2_...",...,0.797595,0.734008,0.089504,3,0.793763,0.871889,0.817467,0.827706,0.032706,27
22,0.778668,0.015083,0.006988,0.000405,0.95,3,10.0,1,20.0,"{'bagging_temperature': 0.95, 'depth': 3, 'l2_...",...,0.797595,0.734008,0.089504,3,0.793763,0.871889,0.817467,0.827706,0.032706,27
27,0.782524,0.014842,0.008551,0.001780,0.01,3,10.0,1,20.0,"{'bagging_temperature': 0.01, 'depth': 3, 'l2_...",...,0.797595,0.734008,0.089504,3,0.793763,0.871889,0.817467,0.827706,0.032706,27
28,0.767649,0.011665,0.008570,0.002220,0.95,3,10.0,1,20.0,"{'bagging_temperature': 0.95, 'depth': 3, 'l2_...",...,0.797595,0.734008,0.089504,3,0.793763,0.871889,0.817467,0.827706,0.032706,27
16,1.548904,0.054029,0.006761,0.000304,0.37729,3,1.0,10,9.551232,"{'bagging_temperature': 0.3772899390185143, 'd...",...,0.812276,0.728292,0.128598,7,0.809813,0.894842,0.832897,0.845851,0.035901,16
24,1.191354,0.004361,0.006926,0.000181,0.04885,3,5.193025,5,20.0,"{'bagging_temperature': 0.048849756429879126, ...",...,0.806826,0.727888,0.125223,8,0.794475,0.876129,0.824549,0.831717,0.033718,24
11,1.949316,0.136924,0.010016,0.000449,0.95,10,10.0,1,20.0,"{'bagging_temperature': 0.95, 'depth': 10, 'l2...",...,0.818708,0.727046,0.127731,9,0.810778,0.899719,0.834965,0.848487,0.037548,13
14,0.968498,0.010913,0.007503,0.000306,0.949755,4,1.0,1,20.0,"{'bagging_temperature': 0.9497551967583575, 'd...",...,0.802890,0.725718,0.127777,10,0.799626,0.883965,0.823386,0.835659,0.035508,20


In [20]:
mt_catboost.train()

✅ Analysis report generated at outputs/model_analysis.html
